# PRABHĀSA — Masked LM Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/PSALM/blob/main/notebooks/demo_01_prabhasa_inference.ipynb)

Load the PRABHĀSA model (`qbz506/prabhasa-b_ss-0.1`) from the Hugging Face Hub and run masked language model fill-in examples. The model uses a custom ELC (Encoder-Language-Core) architecture with BERT-family MLM pretraining, enhanced with Vidyut morpheme-boundary embeddings and Paribhāṣā kāraka-aware adaptive masking.

This notebook is self-contained and runs on the free Colab CPU/GPU runtime with no repository checkout required.

In [ ]:
%pip install -q --upgrade transformers torch sentencepiece

In [ ]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

# PRABHĀSA model: Strict-Small track, 114M parameters, BabyLM pretraining
MODEL_ID = "qbz506/prabhasa-b_ss-0.1"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_ID} on {device}...")

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID, trust_remote_code=True).to(device).eval()

print(f"✓ Model loaded. Vocab size: {tok.vocab_size}")
print(f"  Mask token: {tok.mask_token!r} (id={tok.mask_token_id})")

## Fill-in-the-blank inference

The model predicts the top-k tokens for each `[MASK]` position. We show the softmax probabilities for the true token and the model's top predictions.

In [ ]:
@torch.no_grad()
def fill_in_the_blank(text: str, topk: int = 5):
    """
    Fill-in-the-blank inference: given a text with [MASK] placeholder,
    return the top-k token predictions with probabilities.
    """
    inputs = tok(text, return_tensors="pt").to(device)
    outputs = model(**inputs)
    logits = outputs.logits

    # Find mask token positions
    mask_positions = (inputs.input_ids == tok.mask_token_id).nonzero(as_tuple=True)

    results = []
    for pos_idx in mask_positions[1]:
        mask_logits = logits[0, pos_idx]
        probs = torch.softmax(mask_logits, dim=-1)
        top_probs, top_ids = torch.topk(probs, topk)

        predictions = [
            (tok.decode([tid.item()]), prob.item())
            for tid, prob in zip(top_ids, top_probs, strict=False)
        ]
        results.append(predictions)

    return results


# Example 1: Subject-verb agreement
print("\n" + "=" * 70)
print("Example 1: Subject-verb agreement")
print("=" * 70)
text1 = "The cat [MASK] on the mat."
print(f"Input: {text1}")
preds = fill_in_the_blank(text1, topk=5)
print("\nTop-5 predictions:")
for pred, prob in preds[0]:
    print(f"  {pred:15s} {prob:.4f}")

# Example 2: Object pronouns
print("\n" + "=" * 70)
print("Example 2: Object pronouns")
print("=" * 70)
text2 = "She gave the book to [MASK]."
print(f"Input: {text2}")
preds = fill_in_the_blank(text2, topk=5)
print("\nTop-5 predictions:")
for pred, prob in preds[0]:
    print(f"  {pred:15s} {prob:.4f}")

# Example 3: Tense agreement
print("\n" + "=" * 70)
print("Example 3: Tense agreement")
print("=" * 70)
text3 = "Yesterday I [MASK] to the store."
print(f"Input: {text3}")
preds = fill_in_the_blank(text3, topk=5)
print("\nTop-5 predictions:")
for pred, prob in preds[0]:
    print(f"  {pred:15s} {prob:.4f}")

# Example 4: Multiple masks
print("\n" + "=" * 70)
print("Example 4: Multiple masks (grammatical dependencies)")
print("=" * 70)
text4 = "The [MASK] [MASK] running in the park."
print(f"Input: {text4}")
preds = fill_in_the_blank(text4, topk=3)
print("\nPosition 1 (noun):")
for pred, prob in preds[0]:
    print(f"  {pred:15s} {prob:.4f}")
print("\nPosition 2 (verb):")
for pred, prob in preds[1]:
    print(f"  {pred:15s} {prob:.4f}")

## Tokenization structure

PRABHĀSA uses a SentencePiece 20K BPE tokenizer with morpheme-aware boundary markers. The Vidyut morpheme-boundary embeddings (N-hot layer) assigns morphological roles to each token:

In [ ]:
def show_tokenization(text: str):
    """
    Tokenize and display the token sequence with SentencePiece markers.
    ▁ = word-initial BPE token
    No ▁ = continuation piece
    """
    tokens = tok.tokenize(text)
    ids = tok.encode(text)

    print(f"\nInput: {text!r}")
    print(f"\nTokens ({len(tokens)} total):")
    for i, (tok_str, tok_id) in enumerate(zip(tokens, ids, strict=False)):
        marker = "[WD]" if tok_str.startswith("▁") else "[CT]"
        clean = tok_str.replace("▁", "")
        print(f"  {i:2d}. {marker} {clean:15s} (id={tok_id})")


# Show tokenization for a few examples
print("\n" + "=" * 70)
print("Tokenization: word-initial (WD) vs. continuation (CT) structure")
print("=" * 70)

show_tokenization("The cats are running quickly.")
show_tokenization("I saw her yesterday.")

## Architecture notes

**PRABHĀSA (Arm B, Strict-Small):**
- **Architecture:** ELC (Encoder-Language-Core), 14 transformer layers, 12 attention heads, d_model=768
- **Pretraining:** 10M BabyLM Strict-Small English tokens
- **Mechanisms:**
  - **Vidyut N-hot embeddings:** Morpheme-boundary classification (10-dim: bpe_word_start, bpe_continuation, bpe_suffix_like, bpe_prefix_like, vidyut_root, vidyut_pratyaya, etc.)
  - **Paribhāṣā masking:** kāraka-aware adaptive token masking probability during pretraining
  - **Salience transfer:** learned importance weighting for syntactic features
- **Objective:** Hybrid MLM (30% masking) + CLM at pretraining; zero-shot evaluation on SCAN, COGS, CFQ, BLiMP (grammatical agreement, long-range dependencies, movement, binding, etc.)

See the [results page](https://SharathSPhD.github.io/PSALM/results/) for full evaluation numbers and comparison to baselines.